# Puntos ideales · Cámara de Diputados, 2022–2026

Grid de 10 gráficos (5 años × 2 métodos) a partir de `idps_py/idps.csv`
(generado por [`03_puntos_ideales.py`](03_puntos_ideales.py), que calcula los
puntos ideales con dos métodos: **PCA** (el atajo, primera componente
principal) y **Bayes** (el modelo de Clinton, Jackman y Rivers, implementado
en [`ideal_pscl.py`](ideal_pscl.py)).

Cada panel es un gráfico de barras divergentes: una fila por legislador,
ordenado de izquierda (azul, `idp < 0`) a derecha (rojo, `idp > 0`). En los
paneles de Bayes, una banda gris detrás de la barra muestra `idp_bayes ±
sd_bayes` — la incertidumbre de la estimación. Pasar el mouse sobre cualquier
barra muestra el nombre del legislador y su valor.

In [ ]:
import math

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Cargar los puntos ideales

In [ ]:
df = pd.read_csv("idps_py/idps.csv")
df.head()

## Orientar el PCA

El signo de `idp_pca` es arbitrario por año (lo decide sklearn en
`03_puntos_ideales.py`, no los datos — ver `pca_atajo()`). Lo reorientamos acá
contra `idp_bayes`, que sí viene fijado contra legisladores de referencia de
izquierda/derecha (`orientar()` en `03_puntos_ideales.py`): si la correlación
entre los dos métodos da negativa para un año, se invierte el signo de
`idp_pca` ese año.

In [ ]:
def orientar_pca(df):
    df = df.copy()
    for anio, g in df.groupby("anio"):
        r = g["idp_pca"].corr(g["idp_bayes"])
        if r is not None and r < 0:
            df.loc[g.index, "idp_pca"] = -df.loc[g.index, "idp_pca"]
    return df


df = orientar_pca(df)

## Armar el grid de 10 paneles

In [ ]:
IZQUIERDA, DERECHA = "#2a78d6", "#e34948"
WHISKER, BASELINE, GRID = "#898781", "#c3c2b7", "#e1e0d9"
SURFACE, PAGE = "#fcfcfb", "#f9f9f7"
TEXT, TEXT2, MUTED = "#0b0b0b", "#52514e", "#898781"
FONT = "system-ui, -apple-system, \'Segoe UI\', sans-serif"


def panel(fig, row, col, nombres, valores, sd=None):
    """Un panel: barras divergentes desde 0, ordenadas por valor, con
    opcionalmente un whisker +-sd (Bayes) y hover de nombre + idp."""
    orden = valores.argsort()
    v, nom = valores.to_numpy()[orden], nombres.to_numpy()[orden]
    colores = [DERECHA if x >= 0 else IZQUIERDA for x in v]
    kw = dict(
        x=v, y=list(range(len(v))), orientation="h",
        marker=dict(color=colores, line=dict(width=0)),
        showlegend=False,
        hoverlabel=dict(bgcolor=TEXT, font_color=SURFACE, font_size=12),
    )
    if sd is not None:
        s = sd.to_numpy()[orden]
        kw["error_x"] = dict(type="data", array=s, color=WHISKER, thickness=1, width=0)
        kw["customdata"] = list(zip(nom, s))
        kw["hovertemplate"] = "<b>%{customdata[0]}</b><br>idp = %{x:.2f} (\u00b1%{customdata[1]:.2f})<extra></extra>"
    else:
        kw["customdata"] = nom
        kw["hovertemplate"] = "<b>%{customdata}</b><br>idp = %{x:.2f}<extra></extra>"
    fig.add_trace(go.Bar(**kw), row=row, col=col)

In [ ]:
anios = sorted(df["anio"].unique())

bayes_domain = math.ceil((df["idp_bayes"].abs() + df["sd_bayes"]).max() * 10) / 10
pca_domain = 1.08

fig = make_subplots(
    rows=len(anios), cols=2,
    column_titles=["Bayes · media posterior", "PCA · reescalado a [-1, 1], orientado contra Bayes"],
    row_titles=[str(a) for a in anios],
    horizontal_spacing=0.05, vertical_spacing=0.03,
)

for i, anio in enumerate(anios, start=1):
    g = df[df["anio"] == anio]
    panel(fig, i, 1, g["nombre"], g["idp_bayes"], sd=g["sd_bayes"])
    gp = g.dropna(subset=["idp_pca"])
    panel(fig, i, 2, gp["nombre"], gp["idp_pca"])

# leyenda compartida: trazas invisibles, una sola vez
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
                          marker=dict(size=10, color=IZQUIERDA),
                          name="Izquierda (idp < 0)"), row=1, col=1)
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
                          marker=dict(size=10, color=DERECHA),
                          name="Derecha (idp > 0)"), row=1, col=1)
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
                          line=dict(color=WHISKER, width=2),
                          name="\u00b1 1 desvio (incertidumbre, solo Bayes)"), row=1, col=1)

for i in range(1, len(anios) + 1):
    fig.update_xaxes(range=[-bayes_domain, bayes_domain], zeroline=True,
                      zerolinecolor=BASELINE, zerolinewidth=1, gridcolor=GRID,
                      tickfont=dict(size=10, color=MUTED), row=i, col=1)
    fig.update_xaxes(range=[-pca_domain, pca_domain], zeroline=True,
                      zerolinecolor=BASELINE, zerolinewidth=1, gridcolor=GRID,
                      tickfont=dict(size=10, color=MUTED), row=i, col=2)
    for c in (1, 2):
        fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False,
                          autorange="reversed", row=i, col=c)

fig.update_annotations(font=dict(size=13, color=TEXT2))
for ann in fig.layout.annotations:
    if ann.text in [str(a) for a in anios]:
        ann.font = dict(size=15, color=TEXT)

fig.update_layout(
    title=dict(text="Puntos ideales \u00b7 C\u00e1mara de Diputados, 2022\u20132026", x=0.01,
               font=dict(size=20, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT),
    paper_bgcolor=PAGE, plot_bgcolor=SURFACE,
    height=1500, width=1000,
    margin=dict(l=70, r=20, t=110, b=30),
    legend=dict(x=1.0, y=1.0, xanchor="left", yanchor="top", font=dict(size=11)),
    bargap=0.1,
)

fig.show(config={"displaylogo": False, "scrollZoom": False})